[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C05/E05-1_RUL.ipynb)

# Durée de vie résiduelle d'un équipement : ajuster une tendance et lire les résidus

## Le contexte

Un parc d'organes de transmission est suivi par quatre mesures : le couple de charge, la puissance mécanique, la masse vibrante et le temps de réponse du système. Chaque équipement porte aussi un niveau de sollicitation thermique, faible, moyen ou élevé. Le service maintenance voudrait estimer la durée de vie résiduelle, en jours, de chaque équipement, pour planifier les arrêts avant la panne.

La grandeur à prédire est un nombre de jours : c'est un problème de régression.

Défi : prédire `RUL_jours` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une régression linéaire multiple sur les quatre mesures et le niveau thermique encodé ; il atteint un R² de 0,871 : à vous de lire ses résidus pour comprendre ce qu'il rate, puis de faire mieux. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-RUL.xlsx`, dans le dossier `donnees` du dépôt : 240 équipements, 6 colonnes. Une ligne est un équipement.

| Colonne | Signification | Unité |
|---|---|---|
| `niveau_sollicitation_thermique_cat` | niveau de sollicitation thermique : `Faible`, `Moyen` ou `Élevé` | catégorie |
| `couple_charge_Nm` | couple de charge | N·m |
| `puissance_mecanique_kw` | puissance mécanique | kW |
| `masse_vibrante_kg` | masse vibrante | kg |
| `temps_reponse_ms` | temps de réponse du système | ms |
| `RUL_jours` | cible : durée de vie résiduelle | jours |

Origine : jeu d'enseignement réhabillé en organes de transmission, dont la source n'est pas identifiée. Les variables sont renommées et trois équipements n'ont pas de niveau thermique renseigné. Il enseigne une méthode, et aucune conclusion de maintenance n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition du niveau thermique avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-RUL.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-RUL.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "RUL_jours"
niveau = ["niveau_sollicitation_thermique_cat"]
print("dimensions :", df.shape)
vides = df.isna().sum()
print("cases vides :", vides[vides > 0].to_dict())
print(df[niveau[0]].value_counts(dropna=False).to_dict())
df.head(3)

Lecture du résultat : 240 équipements, 6 colonnes. Le niveau thermique compte 3 cases vides, et les quatre mesures n'en ont aucune. Les équipements les moins sollicités sont les plus nombreux. Un modèle linéaire ne lit que des nombres : le niveau thermique devra être encodé, et ses cases vides remplies.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-RUL.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des équipements est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "équipements")
print("test :", len(X_test), "équipements")

### Étape 3 : construire le modèle de départ

Le modèle de départ est celui de la section du livre : une régression linéaire multiple. Les quatre mesures entrent telles quelles. Le niveau thermique est encodé en colonnes 0 ou 1, après avoir remplacé les cases vides par le niveau le plus fréquent de l'apprentissage. Il apprend sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 4 : mesurer sur la base de test

Le modèle prédit les équipements qu'il n'a jamais vus. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en jours, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin sans apprentissage, qui répond toujours la moyenne de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.1f} jours")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.1f} jours")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")

Lecture du résultat : le R² est de 0,871 et l'erreur absolue moyenne de 14,9 jours, pour une durée de vie moyenne de 200 jours sur la base de test. Le témoin, qui répond toujours la moyenne, n'explique rien : son R² est nul, au signe de l'arrondi près. Le modèle explique l'essentiel de la variance, mais un R² honorable peut cacher une structure qu'il ne voit pas : c'est ce que les résidus montrent. C'est ce chiffre, mesuré sur des équipements que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

### Étape 5 : lire les résidus sur l'apprentissage

Un résidu est l'écart entre la durée observée et la durée prédite. Pour qu'il ne soit pas flatté par l'apprentissage, on le calcule par validation croisée : chaque équipement est prédit par un modèle qui ne l'a pas vu. La lecture se fait sur l'apprentissage seul, par niveau thermique : la moyenne des résidus dit si le modèle se trompe en moyenne dans un groupe, l'écart-type dit s'il y est dispersé.

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict

plis = KFold(n_splits=5, shuffle=True, random_state=GRAINE)
residus = y_train - cross_val_predict(modele, X_train, y_train, cv=plis)
groupe = X_train[niveau[0]].fillna("sans niveau")
resume = residus.groupby(groupe).agg(["count", "mean", "std"])
print(resume.round(1).to_string())
print("durées observées des équipements sans niveau :", y_train[groupe == "sans niveau"].tolist())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.9), sharey=True)
part_expliquee = {}
for ax, (nom, marque) in zip(axes, [("Faible", "o"), ("Moyen", "s"), ("Élevé", "^")]):
    t, r = X_train.loc[groupe == nom, "temps_reponse_ms"], residus[groupe == nom]
    courbe = np.polyfit(t, r, 2)
    part_expliquee[nom] = 1 - ((r - np.polyval(courbe, t)) ** 2).sum() / ((r - r.mean()) ** 2).sum()
    grille = np.linspace(t.min(), t.max(), 50)
    ax.scatter(t, r, s=16, marker=marque, color="0.6", edgecolor="black", linewidth=0.4)
    ax.plot(grille, np.polyval(courbe, grille), color="black", linewidth=1.4)
    ax.axhline(0, color="black", linewidth=0.6, linestyle=":")
    ax.set(title=nom, xlabel="temps de réponse (ms)", ylim=(-90, 130))
    ax.grid(alpha=0.3)
axes[0].set_ylabel("résidu (jours)")
plt.tight_layout()
plt.show()
print("part des résidus décrite par une courbe de degré 2 du temps de réponse :")
print({nom: round(float(valeur), 2) for nom, valeur in part_expliquee.items()})

Lecture du résultat : les résidus ne sont pas répartis au hasard. Les 2 équipements sans niveau ont une durée observée de 1 jour et un résidu moyen de -203 jours : le modèle les imagine en bon état alors qu'ils sont en fin de vie. Les équipements au niveau élevé ont un écart-type de 30 jours, contre 9 au niveau faible. Sur la figure, les résidus n'ont pas la même forme dans les trois niveaux : aucune au niveau faible, une courbe qui monte puis s'aplatit au niveau moyen, une courbe en creux au niveau élevé. Une courbe de degré 2 du temps de réponse décrit 0 % des résidus du niveau faible, 59 % de ceux du niveau moyen et 31 % de ceux du niveau élevé. Deux pistes en sortent : une case vide de niveau peut être une information, et la relation entre le temps de réponse et la durée de vie n'a pas la même forme selon le niveau.

## Pour aller plus loin : un polynôme par niveau thermique

Le modèle de départ de l'étape 3 reste le score de référence du défi. Les deux cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première construit, pour chaque niveau thermique, des termes en puissances du temps de réponse, et compare par validation croisée sur l'apprentissage le modèle de départ, le niveau inconnu traité comme une catégorie à part, un polynôme de degré 2 sur toutes les colonnes et un polynôme par niveau de degré 1, 2 et 3. La seconde mesure le modèle retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 3 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Un R² honorable ne prouve pas que le modèle a compris les données : les résidus montrent ce qu'il rate.
2. Les résidus se lisent sur l'apprentissage seul, par validation croisée, groupe par groupe et contre chaque variable ; le test ne sert qu'à mesurer.
3. Une case vide n'est pas toujours un oubli : ici, elle accompagne des équipements en fin de vie. La remplir par la valeur la plus fréquente efface l'information, la traiter comme une catégorie la garde.
4. Quand la relation change de forme d'un groupe à l'autre, un terme de plus ne suffit pas : il faut une courbe par groupe, c'est-à-dire des interactions.
5. Le choix du modèle se fait par validation croisée sur l'apprentissage ; le test ne sert qu'une fois, au modèle retenu. Sur un test de 60 équipements, un seul équipement en fin de vie, dont le niveau n'est pas renseigné, suffit à faire perdre plusieurs centièmes de R² à un modèle qui l'imagine en bon état.